# 05_05 Validation: did it learn, or memorise?

*📖 Book: section 5.1.3 (training and validation losses)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/homer_model.pt`
>
> **This notebook:** A falling training loss may only mean the model memorised the text. We hide the last 10% of Book 1, measure the loss on it, and see whether the model really learns.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to split Book 1 for real, and check the sizes
- How to write `evaluate()`: the loss, measured without learning
- How to read a training-vs-validation graph
- Why the trained model's score on the last 10% isn't a fair test

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **validation loss** | the loss on the validation set: text the model never trained on | `2.87` |
| **evaluate** | measure the average loss on some text, without changing the model | `evaluate(model, val_data)` |
| **torch.no_grad()** | a block of code where PyTorch doesn't prepare for learning: measure only | `with torch.no_grad():` |
| **perplexity** | `e` to the power of the loss: how many characters the model is choosing between, roughly | loss 2.38 → about 11 |
| **data leak** | test text that the model has already seen in training, so the test isn't fair | the last 10% inside `homer_model.pt`'s training |

## 3. The idea

### ❓ The training loss went down. Did the model LEARN, or did it memorise?

The training loss can't tell us. It's measured on text the model practised on, like a student marking their own homework. In [05_03](05_03_memorising_vs_understanding.ipynb) we saw why (overfitting) and the cure: hide a part of the text, the **validation set**, and only ever test on it. This notebook does it for real, on Book 1, and measures the validation loss.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I make the real split?

With the same two slices as in [05_03](05_03_memorising_vs_understanding.ipynb), on the real data. `load_data` gives all of Book 1 as numbers, and we cut at 90%.

```
Book 1 (19,701 characters)
[ ───────── train_data: data[:n] ─────────── ][ val_data: data[n:] ]
```

👉 **Predict first:** 90% of 19,701 is 17,730.9. What does `int(...)` make of it? How many characters are left for validation?

In [ ]:
import math

import torch
import torch.nn.functional as F

from homer_ai.model import HomerGreekLLM
from homer_ai.train_homer import BOOKS_DIR, load_data

data, tokenizer, text = load_data(BOOKS_DIR)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("all data:  ", len(data))
print("training:  ", len(train_data))
print("validation:", len(val_data))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

17,730 characters to train on and 1,971 to validate on, the numbers we expected. `int` chopped off the `.9`. One thing to hold on to: **the split happens before training, once, and the model never sees `val_data` while it learns.**

</details>

### ❓ Your turn: can you make an 80/20 split?

Some people hold back more than 10%. Cut the same `data` at 80%: the first part to train on, the rest held out.

👉 **Predict first:** 80% of 19,701 is 15,760.8. How long will each part be?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: find the cut point at 80% of the data (a whole number), then cut twice
# 💡 hint: use the same cut point in both slices
cut = None
train_80 = None
held_out_20 = None

In [ ]:
from homer_ai.lessons import check

check(
    "length of the training part",
    train_80,
    data[: int(0.8 * len(data))],
    show=len,
    coach="The training part is everything before the cut: data[:cut], with "
    "cut = int(0.8 * len(data)). A slice needs a whole number, and int() "
    "makes one.",
)
check(
    "length of the held-out part",
    held_out_20,
    data[int(0.8 * len(data)) :],
    show=len,
    coach="The held-out part starts exactly where the training part stops: "
    "data[cut:]. If the two overlap, the model is tested on text it trained on. "
    "If there's a gap between them, text is lost.",
)

<details><summary>💡 Hint</summary>

The cell above does it with `0.9`: `n = int(0.9 * len(data))`, then `data[:n]` and `data[n:]`.

</details>

<details><summary>✅ Answer</summary>

```python
cut = int(0.8 * len(data))
train_80 = data[:cut]
held_out_20 = data[cut:]
```

15,760 characters to train on and 3,941 held out. Add them up and you get 19,701 again: nothing lost, nothing used twice. A bigger held-out part gives a steadier score, but the model has less text to learn from. With a text this small, we keep 10%.

</details>

### ❓ How do I measure the loss without teaching the model?

We write a function, `evaluate`, that does what the training loop does, **minus the learning**. Compare the two:

```
train() loop                    evaluate()
-----------                     ----------
logits = model(inputs)          logits = model(inputs)
loss = cross_entropy(...)       loss = cross_entropy(...)
optimizer.zero_grad()    ✗      (not here)
loss.backward()          ✗      (not here)
optimizer.step()         ✗      (not here)
```

Those three lines are the learning: `zero_grad` clears the previous nudges, `backward` works out fresh ones, `step` applies them. If `evaluate` had them, testing on the validation set would also teach the model the validation set, and the test wouldn't be fair any more.

`with torch.no_grad():` is the second safeguard. It says: measure only, don't learn. Inside it, PyTorch doesn't keep the notes it would need for `backward()`, so it also runs a little faster and uses less memory.

One bit of maths before we run it. Random guessing among 137 characters gives each a chance of 1/137, so the loss ([05_01](05_01_loss_and_epochs.ipynb): minus the log of the chance) is `ln(137)`. `ln` is the **natural logarithm**: `ln(x)` is the power you raise `e` (about 2.718) to, to get `x`. So `ln(137)` is about 4.92, because `e` to the power 4.92 is about 137. (It's `math.log` in Python.)

👉 **Predict first:** an untrained model guesses at random among 137 characters. About what loss should `evaluate` give on the validation text?

In [ ]:
def evaluate(model, data, chunk_size=24):
    """Average loss on `data`. Measure only: the model does not learn."""
    losses = []
    with torch.no_grad():  # "measure only, don't learn"
        for i in range(0, len(data) - chunk_size, chunk_size):
            inputs = data[i : i + chunk_size]
            targets = data[i + 1 : i + chunk_size + 1]
            loss = F.cross_entropy(model(inputs), targets)
            losses.append(loss.item())
    return sum(losses) / len(losses)


torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)

print("untrained, on validation:", round(evaluate(model, val_data), 3))
print("random guessing, ln(137):", round(math.log(tokenizer.vocab_size), 3))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 4.93 for the untrained model, close to `ln(137) = 4.92`. That's pure guessing, and it's our starting line. Notice that `evaluate` never touched the optimizer. There's no `optimizer` in it at all.

</details>

### ❓ What does a real run look like, training loss next to validation loss?

We train our small `HomerGreekLLM` on `train_data` only, for 8 epochs (about 6 seconds). After every epoch we print the training loss (averaged on the way) and the validation loss (from `evaluate`). The seed (see [05_01](05_01_loss_and_epochs.ipynb)) makes it repeatable.

👉 **Predict first:** will the validation loss be lower or higher than the training loss? Will it keep falling?

In [ ]:
torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

train_curve, val_curve = [], []
for epoch in range(8):
    losses = []
    for i in range(0, len(train_data) - 24, 24):
        inputs = train_data[i : i + 24]
        targets = train_data[i + 1 : i + 24 + 1]
        loss = F.cross_entropy(model(inputs), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    train_curve.append(sum(losses) / len(losses))
    val_curve.append(evaluate(model, val_data))
    print(
        f"epoch {epoch + 1} | train {train_curve[-1]:.3f} "
        f"| validation {val_curve[-1]:.3f}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Both losses fall a lot in the first epoch (from about 4.9) and then slowly. The validation loss stays a little **above** the training loss. That's normal, because the model has practised on one and not the other. (In epoch 1 the training number is higher, but only because it's averaged *while* the model is still learning. The validation loss is measured at the end of the epoch.) The graph shows it better.

</details>

### ❓ How do I read the two curves?

```
train ↓  validation ↓   → learning real patterns
train ↓  validation ↑   → memorising (overfitting): the curves split apart
```

So look for two things: the gap between the lines, and which way the validation line goes.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, GOOD, studio_axes, studio_figure

epochs = list(range(1, 9))
fig = studio_figure(
    "training vs validation",
    "Both fall, close together: learning, not memorising",
    size=(9, 5.5),
)
ax = studio_axes(fig, (0.1, 0.14, 0.8, 0.64))
style = {"marker": "o", "linewidth": 2.5}
ax.plot(epochs, train_curve, color=GOOD, label="training loss", **style)
ax.plot(epochs, val_curve, color=ACCENT, label="validation loss", **style)
ax.annotate(
    f"{val_curve[-1]:.2f}", (8, val_curve[-1]), textcoords="offset points",
    xytext=(8, 6), color=ACCENT, fontsize=11, fontfamily="monospace", fontweight="bold",
)
ax.annotate(
    f"{train_curve[-1]:.2f}", (8, train_curve[-1]), textcoords="offset points",
    xytext=(8, -16), color=GOOD, fontsize=11, fontfamily="monospace", fontweight="bold",
)
ax.set_xlim(0.7, 8.9)
ax.set_xlabel("epoch")
ax.set_ylabel("average loss")
ax.legend(frameon=False, labelcolor="white")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

What can we really say? In this run both lines fall and still sit close together, with validation a little above training. That's **learning**, not memorising. We don't claim more than that. 8 epochs is short, so this run can't tell us whether a longer one would start to overfit. (The 5-line model of [05_03](05_03_memorising_vs_understanding.ipynb) had a tiny text to memorise, and it did.) Overfitting would show as the orange line turning upwards while the green one keeps falling, and we don't see that here. That's what the validation loss is for: we can check, and we don't have to hope.

</details>

### ❓ What does the trained `homer_model.pt` score on the last 10%?

We have a real model, trained for 800 epochs: yours (`src/homer_ai/homer_model.pt`) if you trained one, otherwise the ready-made copy in [models/](../../models/README.md). We can load it (read-only, and the next notebook explains how) and run `evaluate` on `val_data`.

👉 **Predict first:** will it be better (lower) than the 8-epoch run above? Is it a fair test?

In [ ]:
from homer_ai.lessons import load_homer_model

# your own homer_model.pt if you made one, otherwise the ready-made model in models/
saved_tokenizer, trained = load_homer_model()

print("trained model, on the first 90%:", round(evaluate(trained, train_data), 3))
print("trained model, on the last 10%: ", round(evaluate(trained, val_data), 3))
print("8-epoch run, on the last 10%:   ", round(val_curve[-1], 3))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 2.38 on the first 90% and about 2.53 on the last 10%, against 2.88 for our 8-epoch run. **But this 2.53 isn't a fair test.** `homer_model.pt` was trained on all of Book 1, including the last 10%, 800 times. That's a data leak. It's like a student sitting an exam they've already seen. How much does the leak flatter the number? We can't say. Maybe only a little, because our model is small. And this is exactly why the split has to happen before training. You can't split afterwards and trust the result.

In short: our 8-epoch run is a fair test of a weaker model. The saved model is a better model, but its score isn't honest.

</details>

### ❓ What does a loss like 2.53 mean in everyday words?

**Perplexity** is `e` to the power of the loss. It says the model is about as unsure as if it were choosing between that many characters. In [05_01](05_01_loss_and_epochs.ipynb), `e^2.38 ≈ 11`. Here:

In [ ]:
perplexity = math.exp(evaluate(trained, val_data))
print("perplexity on the last 10%:", round(perplexity, 1))
print("random guessing among 137:  ", tokenizer.vocab_size)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 12.6. So the model isn't choosing among 137 characters any more. It's about as unsure as if it were choosing between 13. Lower is better.

</details>

### ❓ Is there a model whose held-out score IS fair?

Yes: the **GPT** of [chapter 4](../04_building_the_model/04_05_the_full_model.ipynb). `uv run homer-train --model gpt` does what this notebook describes. It trains on the first 90% of Book 1 only, and prints the loss on the last 10% every 20 epochs. The ready-made copy, `models/odyssey_book1_gpt.pt`, was trained that way (or yours, `src/homer_ai/homer_gpt.pt`, if you ran it). We only load it here (no training), and measure it on `val_data`, which it never saw.

👉 **Predict first:** the GPT has 835,072 weights, about 70 times more than our small model. Will its held-out loss beat the unfair 2.53? And the bigram's 2.99 (a table counting which letter follows which, [README](../../README.md))?

In [ ]:
from homer_ai.lessons import load_homer_gpt
from homer_ai.train_homer import evaluate_gpt

gpt_tokenizer, gpt = load_homer_gpt()  # read-only
assert gpt_tokenizer.chars == tokenizer.chars  # the same numbers for the same letters

gpt_train = evaluate_gpt(gpt, train_data)
gpt_val = evaluate_gpt(gpt, val_data)
print("GPT, on the first 90% (it trained on it):", round(gpt_train, 3))
print("GPT, on the last 10%  (it never saw it): ", round(gpt_val, 3))
print("perplexity on the last 10%:", round(math.exp(gpt_val), 1))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 1.51 on the text it trained on and **2.27 on the last 10%**, a perplexity of about 9.7. This one is a fair, held-out number. It's better than the bigram's 2.99, much better than random guessing (4.92), and even lower than the small model's leaked 2.53. The gap between 1.51 and 2.27 is the price of never having seen the text. Some memorising is always left. It's still a long way from a model that writes Greek sentences, but the numbers are real. The samples are in [06_01](../06_making_it_talk/06_01_writing_greek.ipynb) and [06_02](../06_making_it_talk/06_02_less_repetition.ipynb).

</details>

### ❓ Why didn't the GPT's training overfit badly?

It could have. With 835,072 weights and only 17,730 letters, a model can learn the text by heart (the 5-line model of [05_03](05_03_memorising_vs_understanding.ipynb) did). Two things held it back: **dropout** (0.3: while training, 3 numbers in 10 are switched off at random, so it can't rely on any single one), and a short run (200 epochs, with the learning rate sliding down to 0). The held-out line printed during training is how you'd know when to stop. [05_07](05_07_train_the_gpt.ipynb) walks through that training loop, and shows on a small slice what happens to the held-out loss without dropout.

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Should `train()` print the validation loss too?

Yes. That's how you'd spot overfitting while it happens, and stop early. The small model's `train()` doesn't do it, to keep it simple. The GPT's `train_gpt()` in `train_homer.py` does, every 20 epochs.

### ❓ Does `torch.no_grad()` change the model?

No. `no_grad` only stops PyTorch from taking notes for learning. The weights are the same before and after.

## 5. Recap

- Split first (`data[:n]` and `data[n:]`: 17,730 / 1,971), train on the first part only, test on the second.
- `evaluate` = the training loop without `zero_grad`, `backward` and `step`, inside `torch.no_grad()`.
- Train and validation both falling = learning. A saved model that saw the validation text gives an unfair score.

**You can now:**

- split data into a training part and a held-out part, with no overlap
- measure a loss without learning from it: `torch.no_grad()`, and no `step`
- read a training curve next to a validation curve

**Next:** [05_06: Saving and loading](05_06_saving_and_loading.ipynb): how to keep the trained model, and use it later.